Reading bronze table

In [0]:
bronze_df= spark.table(
    "workspace.default.bronze_transactions"
)
display(bronze_df.limit(10))

step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,ingestion_timestamp,source_system
1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,2026-06-22T22:02:32.430Z,paysim
1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,2026-06-22T22:02:32.430Z,paysim
1,TRANSFER,181.0,C1305486145,181.0,0.0,C553264065,0.0,0.0,1,0,2026-06-22T22:02:32.430Z,paysim
1,CASH_OUT,181.0,C840083671,181.0,0.0,C38997010,21182.0,0.0,1,0,2026-06-22T22:02:32.430Z,paysim
1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,2026-06-22T22:02:32.430Z,paysim
1,PAYMENT,7817.71,C90045638,53860.0,46042.29,M573487274,0.0,0.0,0,0,2026-06-22T22:02:32.430Z,paysim
1,PAYMENT,7107.77,C154988899,183195.0,176087.23,M408069119,0.0,0.0,0,0,2026-06-22T22:02:32.430Z,paysim
1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.0,0,0,2026-06-22T22:02:32.430Z,paysim
1,PAYMENT,4024.36,C1265012928,2671.0,0.0,M1176932104,0.0,0.0,0,0,2026-06-22T22:02:32.430Z,paysim
1,DEBIT,5337.77,C712410124,41720.0,36382.23,C195600860,41898.0,40348.79,0,0,2026-06-22T22:02:32.430Z,paysim


In [0]:
bronze_df.printSchema()

root
 |-- step: long (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: long (nullable = true)
 |-- isFlaggedFraud: long (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)



Renaming columns

In [0]:
from pyspark.sql.functions import col

silver_df = (
    bronze_df
    .withColumnRenamed("type", "transaction_type")
    .withColumnRenamed("nameOrig", "source_account")
    .withColumnRenamed("nameDest", "destination_account")
    .withColumnRenamed("oldbalanceOrg", "old_source_balance")
    .withColumnRenamed("newbalanceOrig", "new_source_balance")
    .withColumnRenamed("oldbalanceDest", "old_destination_balance")
    .withColumnRenamed("newbalanceDest", "new_destination_balance")
    .withColumnRenamed("isFraud", "fraud_flag")
    .withColumnRenamed("isFlaggedFraud", "system_fraud_flag")
)

Data Quality Checks
Null Handling

In [0]:
from pyspark.sql.functions import count, when

silver_df.select(
    count(when(col("source_account").isNull(), 1)).alias("null_source_accounts"),
    count(when(col("destination_account").isNull(), 1)).alias("null_destination_accounts")
).show()

+--------------------+-------------------------+
|null_source_accounts|null_destination_accounts|
+--------------------+-------------------------+
|                   0|                        0|
+--------------------+-------------------------+



Negative Amount Check

In [0]:
bronze_df.filter(
    col("amount") < 0
).count()


0

Creating silver transactions table 

In [0]:
(
    silver_df.write
  .format("delta")
  .mode("overwrite")
  .saveAsTable(
      "workspace.default.silver_transactions"
      )
)

Validation

In [0]:
spark.sql("""
          SELECT * 
          FROM workspace.default.silver_transactions
          LIMIT 10 """).show()  

+----+----------------+--------+--------------+------------------+------------------+-------------------+-----------------------+-----------------------+----------+-----------------+--------------------+-------------+
|step|transaction_type|  amount|source_account|old_source_balance|new_source_balance|destination_account|old_destination_balance|new_destination_balance|fraud_flag|system_fraud_flag| ingestion_timestamp|source_system|
+----+----------------+--------+--------------+------------------+------------------+-------------------+-----------------------+-----------------------+----------+-----------------+--------------------+-------------+
|   1|         PAYMENT| 9839.64|   C1231006815|          170136.0|         160296.36|        M1979787155|                    0.0|                    0.0|         0|                0|2026-06-22 22:02:...|       paysim|
|   1|         PAYMENT| 1864.28|   C1666544295|           21249.0|          19384.72|        M2044282225|                    0.0

Row Count checks

In [0]:
spark.sql("""
          SELECT COUNT(*) 
          FROM workspace.default.silver_transactions
          """).show()

+--------+
|COUNT(*)|
+--------+
| 6362620|
+--------+



Schema check


In [0]:
spark.table(
    "workspace.default.silver_transactions"
            ).printSchema()

root
 |-- step: long (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- source_account: string (nullable = true)
 |-- old_source_balance: double (nullable = true)
 |-- new_source_balance: double (nullable = true)
 |-- destination_account: string (nullable = true)
 |-- old_destination_balance: double (nullable = true)
 |-- new_destination_balance: double (nullable = true)
 |-- fraud_flag: long (nullable = true)
 |-- system_fraud_flag: long (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)



Fraud Distribution check

In [0]:
spark.sql("""
            SELECT fraud_flag, COUNT(*)
            FROM workspace.default.silver_transactions
            GROUP BY fraud_flag
            """).show()

+----------+--------+
|fraud_flag|COUNT(*)|
+----------+--------+
|         1|    8213|
|         0| 6354407|
+----------+--------+



Creating silver_accounts

In [0]:
silver_df = spark.table(
    "workspace.default.silver_transactions"
)

Extracting source and destination accounts

In [0]:
source_accounts = silver_df.select("source_account").distinct()
destination_accounts = silver_df.select("destination_account").distinct()

Union both source and destination accounts ,so that we have every account that participated in a transaction.

In [0]:
accounts_df = source_accounts.union(
    destination_accounts
)

Rename column 

In [0]:
from pyspark.sql.functions import lit

accounts_df = (
    accounts_df
    .withColumnRenamed(
        "source_account",
        "account_id"
    )
    .withColumn(
        "account_status",
        lit("ACTIVE")
    )
)

Saving as sliver_table

In [0]:
accounts_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.silver_accounts"
    )

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM workspace.default.silver_accounts
""").show()

+--------+
|COUNT(*)|
+--------+
| 9075669|
+--------+



In [0]:
spark.sql("""
SELECT *
FROM workspace.default.silver_accounts
LIMIT 10
""").show()

+-----------+--------------+
| account_id|account_status|
+-----------+--------------+
|C1144605126|        ACTIVE|
|C1809179400|        ACTIVE|
|C1946536364|        ACTIVE|
| C699085724|        ACTIVE|
|C1580366869|        ACTIVE|
|C2035134131|        ACTIVE|
|C1571387524|        ACTIVE|
|C1558459883|        ACTIVE|
| C157205194|        ACTIVE|
| C550514738|        ACTIVE|
+-----------+--------------+



silver_fraud_transactions

In [0]:
silver_df = spark.table(
    "workspace.default.silver_transactions"
)

In [0]:
from pyspark.sql.functions import col 
fraud_df = silver_df.filter(
    col("fraud_flag") == 1
)

In [0]:
fraud_df.count()

8213

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM workspace.default.silver_transactions
WHERE fraud_flag = 1
""").show()

+--------+
|COUNT(*)|
+--------+
|    8213|
+--------+



In [0]:
(
    fraud_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(
                "workspace.default.silver_fraud_transactions"
            )
)

In [0]:
spark.sql("""
SELECT *
FROM workspace.default.silver_fraud_transactions
LIMIT 10
""").show()

+----+----------------+----------+--------------+------------------+------------------+-------------------+-----------------------+-----------------------+----------+-----------------+--------------------+-------------+
|step|transaction_type|    amount|source_account|old_source_balance|new_source_balance|destination_account|old_destination_balance|new_destination_balance|fraud_flag|system_fraud_flag| ingestion_timestamp|source_system|
+----+----------------+----------+--------------+------------------+------------------+-------------------+-----------------------+-----------------------+----------+-----------------+--------------------+-------------+
|   1|        TRANSFER|     181.0|   C1305486145|             181.0|               0.0|         C553264065|                    0.0|                    0.0|         1|                0|2026-06-22 22:02:...|       paysim|
|   1|        CASH_OUT|     181.0|    C840083671|             181.0|               0.0|          C38997010|             

total number of transactions and the total transaction amount for each transaction type

In [0]:
%sql
SELECT
    transaction_type,
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_transaction_amount,
    AVG(amount) AS average_transaction_amount
FROM workspace.default.silver_transactions
GROUP BY transaction_type;

transaction_type,transaction_count,total_transaction_amount,average_transaction_amount
CASH_OUT,2237500,3.9441299522448804E11,176273.9643461399
PAYMENT,2151495,2.809337113837018E10,13057.604660187533
TRANSFER,532909,4.852919872631636E11,910647.0096454809
DEBIT,41432,2.2719922127999973E8,5483.66531376713
CASH_IN,1399284,2.3636739191245947E11,168920.242004096


In [0]:
from pyspark.sql.functions import count, sum, avg

gold_transaction_summary = (
    spark.table("workspace.default.silver_transactions")
         .groupBy("transaction_type")
         .agg(
             count("*").alias("transaction_count"),
             sum("amount").alias("total_transaction_amount"),
             avg("amount").alias("average_transaction_amount")
         )
)

In [0]:
(
    gold_transaction_summary.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.default.gold_transaction_summary")
)

validation

In [0]:
spark.sql("""
SELECT *
FROM workspace.default.gold_transaction_summary
ORDER BY transaction_count DESC
""").show()

+----------------+-----------------+------------------------+--------------------------+
|transaction_type|transaction_count|total_transaction_amount|average_transaction_amount|
+----------------+-----------------+------------------------+--------------------------+
|        CASH_OUT|          2237500|    3.944129952244880...|         176273.9643461399|
|         PAYMENT|          2151495|    2.809337113837018E10|        13057.604660187533|
|         CASH_IN|          1399284|    2.363673919124594...|          168920.242004096|
|        TRANSFER|           532909|    4.852919872631636E11|         910647.0096454809|
|           DEBIT|            41432|    2.2719922127999973E8|          5483.66531376713|
+----------------+-----------------+------------------------+--------------------------+



How many fraudulent transactions occurred?
What is the total fraud amount?
What is the average fraud amount?
Which transaction types have the most fraud?
What percentage of fraudulent transactions comes from each transaction type?

In [0]:
%sql
SELECT
    transaction_type,
    COUNT(*) AS fraud_transaction_count,
    SUM(amount) AS total_fraud_amount,
    AVG(amount) AS average_fraud_amount
FROM workspace.default.silver_fraud_transactions
GROUP BY transaction_type;

transaction_type,fraud_transaction_count,total_fraud_amount,average_fraud_amount
CASH_OUT,4116,5.989202243829998E9,1455102.5859645281
TRANSFER,4097,6.06721318401E9,1480891.6729338542


In [0]:
from pyspark.sql.functions import count, sum, avg

gold_fraud_summary = (
    spark.table("workspace.default.silver_fraud_transactions")
         .groupBy("transaction_type")
         .agg(
             count("*").alias("fraud_transaction_count"),
             sum("amount").alias("total_fraud_amount"),
             avg("amount").alias("average_fraud_amount")
         )
)

In [0]:
(
    gold_fraud_summary.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.default.gold_fraud_summary")
)

validation

In [0]:
spark.sql("""
SELECT *
FROM workspace.default.gold_fraud_summary
ORDER BY total_fraud_amount DESC
""").show()

+----------------+-----------------------+-------------------+--------------------+
|transaction_type|fraud_transaction_count| total_fraud_amount|average_fraud_amount|
+----------------+-----------------------+-------------------+--------------------+
|        TRANSFER|                   4097|    6.06721318401E9|  1480891.6729338542|
|        CASH_OUT|                   4116|5.989202243829998E9|  1455102.5859645281|
+----------------+-----------------------+-------------------+--------------------+



Total money sent.
Number of outgoing transactions.
Fraudulent transactions initiated.

In [0]:
%sql
SELECT
    source_account,
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_sent_amount,
    SUM(fraud_flag) AS fraud_transaction_count
FROM workspace.default.silver_transactions
GROUP BY source_account;

source_account,transaction_count,total_sent_amount,fraud_transaction_count
C1194651228,1,91247.53,0
C1262530038,1,430612.24,0
C1731636253,1,43215.13,0
C1992591945,1,284816.2,0
C1108901964,1,479084.03,0
C418081740,1,97394.53,0
C1054850789,1,270957.68,0
C1512360270,1,204246.68,0
C607546580,1,533.41,0
C286322589,1,240840.11,0


join with accounts

In [0]:
%sql
SELECT
    a.account_id,
    a.account_status,
    t.transaction_count,
    t.total_sent_amount,
    t.fraud_transaction_count
FROM workspace.default.silver_accounts a
LEFT JOIN (

    SELECT
        source_account,
        COUNT(*) AS transaction_count,
        SUM(amount) AS total_sent_amount,
        SUM(fraud_flag) AS fraud_transaction_count
    FROM workspace.default.silver_transactions
    GROUP BY source_account

) t

ON a.account_id = t.source_account;

account_id,account_status,transaction_count,total_sent_amount,fraud_transaction_count
C118248897,ACTIVE,1,142033.35,0
C1644428633,ACTIVE,1,138869.27,0
C781071508,ACTIVE,1,10264.33,0
C991033509,ACTIVE,1,16997.16,0
C1222361382,ACTIVE,1,123136.35,0
C1013433918,ACTIVE,1,99001.7,0
C1864063255,ACTIVE,1,181056.22,0
C1267449310,ACTIVE,1,178084.88,0
C2067922392,ACTIVE,1,5769.02,0
C1555820771,ACTIVE,1,156601.6,0


In [0]:
from pyspark.sql.functions import count, sum

accounts_df = spark.table("workspace.default.silver_accounts")
transactions_df = spark.table("workspace.default.silver_transactions")

transaction_summary = (
    transactions_df
        .groupBy("source_account")
        .agg(
            count("*").alias("transaction_count"),
            sum("amount").alias("total_sent_amount"),
            sum("fraud_flag").alias("fraud_transaction_count")
        )
)

gold_account_summary = (
    accounts_df.join(
        transaction_summary,
        accounts_df.account_id == transaction_summary.source_account,
        "left"
    )
)

In [0]:
(
    gold_account_summary.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.default.gold_account_summary")
)

In [0]:
spark.sql("""
SELECT *
FROM workspace.default.gold_account_summary
LIMIT 20
""").show()

+-----------+--------------+--------------+-----------------+-----------------+-----------------------+
| account_id|account_status|source_account|transaction_count|total_sent_amount|fraud_transaction_count|
+-----------+--------------+--------------+-----------------+-----------------+-----------------------+
|C1143110513|        ACTIVE|   C1143110513|                1|         53007.32|                      0|
| C942338448|        ACTIVE|    C942338448|                1|        208624.87|                      0|
| C429413744|        ACTIVE|    C429413744|                1|        149468.94|                      0|
| C414684403|        ACTIVE|    C414684403|                1|         18627.95|                      0|
|C1204446361|        ACTIVE|   C1204446361|                1|         24457.76|                      0|
|C1778045793|        ACTIVE|   C1778045793|                1|        248043.99|                      1|
| C508785029|        ACTIVE|    C508785029|                1|   